<a href="https://colab.research.google.com/github/BTrifonov/BTrifonov/blob/main/snn_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required packages

In [3]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 2.1 MB/s eta 0:00:00


In [4]:
# imports
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import numpy as np

# Reproducibility
torch.manual_seed(0)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [10]:
#@title Plotting Settings
def plot_cur_mem_spk(cur, mem, spk, thr_line=False, vline=False, title=False, ylim_max1=1.25, ylim_max2=1.25):
  # Generate Plots
  fig, ax = plt.subplots(3, figsize=(8,6), sharex=True,
                        gridspec_kw = {'height_ratios': [1, 1, 0.4]})

  # Plot input current
  ax[0].plot(cur, c="tab:orange")
  ax[0].set_ylim([0, ylim_max1])
  ax[0].set_xlim([0, 200])
  ax[0].set_ylabel("Input Current ($I_{in}$)")
  if title:
    ax[0].set_title(title)

  # Plot membrane potential
  ax[1].plot(mem)
  ax[1].set_ylim([0, ylim_max2])
  ax[1].set_ylabel("Membrane Potential ($U_{mem}$)")
  if thr_line:
    ax[1].axhline(y=thr_line, alpha=0.25, linestyle="dashed", c="black", linewidth=2)
  plt.xlabel("Time step")

  # Plot output spike using spikeplot
  splt.raster(spk, ax[2], s=400, c="black", marker="|")
  if vline:
    ax[2].axvline(x=vline, ymin=0, ymax=6.75, alpha = 0.15, linestyle="dashed", c="black", linewidth=2, zorder=0, clip_on=False)
  plt.ylabel("Output spikes")
  plt.yticks([])

  plt.show()

def plot_snn_spikes(spk_in, spk1_rec, spk2_rec, title):
  # Generate Plots
  fig, ax = plt.subplots(3, figsize=(8,7), sharex=True,
                        gridspec_kw = {'height_ratios': [1, 1, 0.4]})

  # Plot input spikes
  splt.raster(spk_in[:,0], ax[0], s=0.03, c="black")
  ax[0].set_ylabel("Input Spikes")
  ax[0].set_title(title)

  # Plot hidden layer spikes
  splt.raster(spk1_rec.reshape(num_steps, -1), ax[1], s = 0.05, c="black")
  ax[1].set_ylabel("Hidden Layer")

  # Plot output spikes
  splt.raster(spk2_rec.reshape(num_steps, -1), ax[2], c="black", marker="|")
  ax[2].set_ylabel("Output Spikes")
  ax[2].set_ylim([0, 10])

  plt.show()


def plot_single_neuron(current, spk_rec, mem_rec, title="Single LIF neuron"):
    T = len(current)
    t_axis = torch.arange(T)

    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

    # Input current
    axes[0].plot(t_axis, current.numpy())
    axes[0].set_ylabel("Input current")
    axes[0].set_title(title)

    # Membrane potential
    axes[1].plot(t_axis, mem_rec.numpy())
    axes[1].set_ylabel("Membrane")

    # Spikes
    spike_times = t_axis[spk_rec > 0]
    axes[2].scatter(spike_times.numpy(), torch.ones_like(spike_times).numpy(), marker="|", s=200)
    axes[2].set_ylabel("Spikes")
    axes[2].set_xlabel("Time step")
    axes[2].set_ylim(0.8, 1.2)

    plt.tight_layout()
    plt.show()

def plot_two_stream_test(basal, apical, total, spk_rec, mem_rec, title="Basal / Apical Test"):
    T = len(basal)
    t_axis = torch.arange(T)

    fig, axes = plt.subplots(5, 1, figsize=(10, 8), sharex=True)

    axes[0].plot(t_axis, basal.numpy())
    axes[0].set_ylabel("Basal")
    axes[0].set_title(title)

    axes[1].plot(t_axis, apical.numpy())
    axes[1].set_ylabel("Apical")

    axes[2].plot(t_axis, total.numpy())
    axes[2].set_ylabel("Total")

    axes[3].plot(t_axis, mem_rec.numpy())
    axes[3].set_ylabel("Mem")

    spike_times = t_axis[spk_rec > 0]
    axes[4].scatter(spike_times.numpy(), torch.ones_like(spike_times).numpy(), marker="|", s=200)
    axes[4].set_ylabel("Spikes")
    axes[4].set_xlabel("Time step")
    axes[4].set_ylim(0.8, 1.2)

    plt.tight_layout()
    plt.show()

# Spike Data Generation

In [34]:
def generate_random_spikes(
    n_samples: int,
    num_steps: int,
    n_inputs: int,
    max_rate: float = 1.0,
    seed: int = 0,
):
    """
    Generate random spike-train samples using Poisson/Bernoulli rate encoding.

    Random firing probabilities in [0, max_rate] are generated and converted
    into spikes using snnTorch's `spikegen.rate_conv`. Each sample has shape
    [time_steps, batch_size, input_neurons].

    Parameters
    ----------
    n_samples : int
        Number of spike-train samples to generate.
    num_steps : int
        Number of simulation time steps.
    n_inputs : int
        Number of input neurons.
    max_rate : float, optional
        Maximum firing probability per timestep (default = 1.0).
    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    torch.Tensor
        Tensor of shape [n_samples, num_steps, 1, n_inputs].
    """

    torch.manual_seed(seed)
    spikes = []

    for _ in range(n_samples):
        spike_prob = torch.rand(num_steps, n_inputs) * max_rate
        spk = spikegen.rate_conv(spike_prob).unsqueeze(1)  # [T, 1, N]
        spikes.append(spk)

    return torch.stack(spikes)

# Network

## Autoencoder with Difference Target Propagation (DTP)

In [12]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, beta):
        super().__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.reconstruction_dim = input_dim
        self.beta = beta

        # Encoder: input -> latent
        self.encoder = nn.Linear(self.input_dim, self.latent_dim)
        self.latent_neurons = snn.Leaky(beta=self.beta)

        # Decoder: latent -> reconstruction
        self.decoder = nn.Linear(self.latent_dim, self.reconstruction_dim)
        self.output_neurons = snn.Leaky(beta=self.beta)

        # Feedback mapping f: reconstruction/output -> latent
        self.feedback = nn.Linear(self.reconstruction_dim, self.latent_dim, bias=False)

    def forward(self, x):
        """
        Run the spiking autoencoder forward in time.

        Parameters
        ----------
        x : torch.Tensor
            Input spike tensor of shape [time_steps, batch_size, input_dim].

        Returns
        -------
        latent_spikes : torch.Tensor
            Hidden/latent spikes of shape [time_steps, batch_size, latent_dim].

        latent_mem : torch.Tensor
            Hidden/latent membrane potentials of shape [time_steps, batch_size, latent_dim].

        recon_spikes : torch.Tensor
            Output/reconstruction spikes of shape [time_steps, batch_size, reconstruction_dim].

        recon_mem : torch.Tensor
            Output/reconstruction membrane potentials of shape [time_steps, batch_size, reconstruction_dim].
        """
        latent_mem_t = self.latent_neurons.init_leaky()
        recon_mem_t = self.output_neurons.init_leaky()

        latent_spike_history = []
        latent_mem_history = []
        recon_spike_history = []
        recon_mem_history = []

        num_steps = x.size(0)

        for t in range(num_steps):
            # Encode input into latent representation
            latent_current = self.encoder(x[t])
            latent_spikes_t, latent_mem_t = self.latent_neurons(latent_current, latent_mem_t)

            # Decode latent representation back into reconstruction
            recon_current = self.decoder(latent_spikes_t)
            recon_spikes_t, recon_mem_t = self.output_neurons(recon_current, recon_mem_t)

            latent_spike_history.append(latent_spikes_t)
            latent_mem_history.append(latent_mem_t)
            recon_spike_history.append(recon_spikes_t)
            recon_mem_history.append(recon_mem_t)

        latent_spikes = torch.stack(latent_spike_history, dim=0)
        latent_mem = torch.stack(latent_mem_history, dim=0)
        recon_spikes = torch.stack(recon_spike_history, dim=0)
        recon_mem = torch.stack(recon_mem_history, dim=0)

        return latent_spikes, latent_mem, recon_spikes, recon_mem

    def compute_targets(self, x):
        """
        Compute difference target propagation (DTP) targets.

        Output target:
            z_hat = x

        Latent target:
            h_hat = h + f(z_hat) - f(z)

        where
            h     = current latent state
            z     = current reconstruction/output state
            f(.)  = feedback mapping from output space to latent space

        Parameters
        ----------
        x : torch.Tensor
            Input spike tensor of shape [time_steps, batch_size, input_dim].

        Returns
        -------
        latent_spikes : torch.Tensor
        latent_mem : torch.Tensor
        recon_spikes : torch.Tensor
        recon_mem : torch.Tensor
        latent_target : torch.Tensor
            DTP target for the latent state, shape [time_steps, batch_size, latent_dim].
        output_target : torch.Tensor
            Reconstruction target, equal to the input x, shape [time_steps, batch_size, input_dim].
        """
        latent_spikes, latent_mem, recon_spikes, recon_mem = self.forward(x)

        output_target = x.float()
        current_output = recon_mem
        current_latent = latent_mem

        latent_target = (
            current_latent
            + self.feedback(output_target)
            - self.feedback(current_output)
        )

        return (
            latent_spikes,
            latent_mem,
            recon_spikes,
            recon_mem,
            latent_target,
            output_target,
        )

    def train_epoch(
        self,
        dataloader,
        optimizer,
        device,
        num_steps,
        latent_loss_weight=0.1,
        recon_loss_fn=None,
        latent_loss_fn=None,
    ):
        """
        Train the model for one epoch.

        Parameters
        ----------
        dataloader : DataLoader
            Yields MNIST batches as [batch_size, 1, 28, 28].
        optimizer : torch.optim.Optimizer
            Optimizer used for parameter updates.
        device : torch.device
            CPU or CUDA device.
        num_steps : int
            Number of spike simulation steps.
        latent_loss_weight : float, optional
            Weight of the latent DTP loss term.
        recon_loss_fn : callable, optional
            Reconstruction loss function. Defaults to L1 loss.
        latent_loss_fn : callable, optional
            Latent alignment loss function. Defaults to MSE loss.

        Returns
        -------
        dict
            Mean total, reconstruction, and latent losses for the epoch.
        """
        self.train()

        if recon_loss_fn is None:
            recon_loss_fn = nn.L1Loss()
        if latent_loss_fn is None:
            latent_loss_fn = nn.MSELoss()

        total_loss = 0.0
        total_recon = 0.0
        total_latent = 0.0

        for images, _ in dataloader:
            images = images.to(device)

            # Static MNIST -> spike trains
            spike_data = spikegen.rate(images, num_steps=num_steps)

            # [T, B, 1, 28, 28] -> [T, B, 784]
            spike_data = spike_data.view(num_steps, images.size(0), -1).to(device)

            optimizer.zero_grad()

            _, latent_mem, _, recon_mem, latent_target, output_target = self.compute_targets(spike_data)

            loss_recon = recon_loss_fn(recon_mem, output_target)
            loss_latent = latent_loss_fn(latent_mem, latent_target.detach())
            loss = loss_recon + latent_loss_weight * loss_latent

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += loss_recon.item()
            total_latent += loss_latent.item()

        n_batches = len(dataloader)

        return {
            "loss": total_loss / n_batches,
            "recon_loss": total_recon / n_batches,
            "latent_loss": total_latent / n_batches,
        }

In [13]:
# Training Parameters
batch_size = 128
data_path = "/tmp/data/mnist"
num_classes = 10  # MNIST has 10 output classes

# Torch Variables
dtype = torch.float

In [14]:
from torchvision import datasets, transforms
from snntorch import utils
from torch.utils.data import DataLoader

# Define a transform (same as Tutorial 1)
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    transforms.Normalize((0,), (1,))
])

mnist_train = datasets.MNIST(
    data_path,
    train=True,
    download=True,
    transform=transform
)

subset = 10
mnist_train = utils.data_subset(mnist_train, subset)

print(f"The size of mnist_train is {len(mnist_train)}")

train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)

The size of mnist_train is 6000


## Instantiate network

In [15]:
model = Autoencoder(input_dim=784, latent_dim=128, beta=0.95).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_steps = 20
num_epochs = 3

for epoch in range(num_epochs):
    metrics = model.train_epoch(
        dataloader=train_loader,
        optimizer=optimizer,
        device=device,
        num_steps=num_steps,
        latent_loss_weight=0.1,
    )

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"loss={metrics['loss']:.4f} | "
        f"recon={metrics['recon_loss']:.4f} | "
        f"latent={metrics['latent_loss']:.4f}"
    )

Epoch 1/3 | loss=0.2858 | recon=0.2797 | latent=0.0617
Epoch 2/3 | loss=0.1769 | recon=0.1729 | latent=0.0395
Epoch 3/3 | loss=0.1433 | recon=0.1397 | latent=0.0361
